# Submagic — Chạy trên Google Colab GPU T4 (Đồng bộ trực tiếp qua GitHub)

### Cách dùng
1. Trên menu Colab: chọn **Runtime → Change runtime type → Hardware accelerator: `T4 GPU` → Save**.
2. Bấm nút **▶ Chạy (Run)** ở ô **Bước 1** bên dưới:
   - Tự động kết nối Google Drive (giữ lại video, database `sub_magic.sqlite3` và model giữa các phiên).
   - Tự động **`git clone` / `git pull`** code mới nhất từ GitHub.
   - Tự cài môi trường **Python 3.11 + GPU T4** (**PP-OCRv5 + VietOCR + Faster-Whisper**), build giao diện React và mở link web `https://...trycloudflare.com`.

In [ ]:
#@title 🚀 BƯỚC 1: Đồng bộ Git + Khởi động Submagic (Backend + Frontend + GPU T4)
#@markdown Cấu hình Git & thư mục lưu trên Google Drive:
GIT_REPO = "https://github.com/truongvanvuong/Submagic.git"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
AUTO_GIT_PULL = True  #@param {type:"boolean"}
PROJECT_DIR = "/content/drive/MyDrive/Submagic"  #@param {type:"string"}
PORT = 8000  #@param {type:"integer"}

import os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

# 1. Kiểm tra GPU T4
print("=== 1. Kiểm tra GPU ===")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.")
gpu_res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_res.returncode != 0 or not gpu_res.stdout.strip():
    raise RuntimeError("Chưa bật GPU! Vui lòng vào Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.")
print("✅ GPU sẵn sàng:", gpu_res.stdout.strip())

# 2. Kết nối Google Drive & đồng bộ code từ GitHub
print("\n=== 2. Kết nối Google Drive & Đồng bộ GitHub ===")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

auth_repo = GIT_REPO.strip()
if GITHUB_TOKEN.strip() and auth_repo.startswith("https://"):
    auth_repo = auth_repo.replace("https://", f"https://{GITHUB_TOKEN.strip()}@", 1)

proj_path = Path(PROJECT_DIR)
frontend_changed = False
if (proj_path / ".git").exists():
    if AUTO_GIT_PULL:
        print("🔄 Đang kéo code mới nhất (git pull)...")
        old_rev = subprocess.check_output(["git", "-C", str(proj_path), "rev-parse", "HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(proj_path), "pull", "--ff-only"], check=True)
        new_rev = subprocess.check_output(["git", "-C", str(proj_path), "rev-parse", "HEAD"], text=True).strip()
        if old_rev != new_rev:
            diff_out = subprocess.check_output(["git", "-C", str(proj_path), "diff", "--name-only", old_rev, new_rev], text=True)
            frontend_changed = any(line.startswith("frontend/") for line in diff_out.splitlines())
            print(f"✅ Đã cập nhật {old_rev[:7]} -> {new_rev[:7]} (frontend_changed={frontend_changed})")
        else:
            print("✅ Code đã là bản mới nhất.")
elif proj_path.exists():
    # Thư mục đã có sẵn từ lần giải nén zip cũ (chưa có .git) -> khởi tạo git tại chỗ, giữ nguyên backend/data
    print("🔗 Đang gắn Git vào thư mục hiện có (giữ nguyên dữ liệu & video)...")
    subprocess.run(["git", "-C", str(proj_path), "init"], check=True)
    subprocess.run(["git", "-C", str(proj_path), "remote", "add", "origin", auth_repo], check=False)
    subprocess.run(["git", "-C", str(proj_path), "fetch", "origin"], check=True)
    branch = subprocess.check_output(["git", "-C", str(proj_path), "symbolic-ref", "refs/remotes/origin/HEAD"], text=True).strip().split("/")[-1]
    subprocess.run(["git", "-C", str(proj_path), "reset", "--hard", f"origin/{branch}"], check=True)
    frontend_changed = True
else:
    print(f"📥 Đang clone {GIT_REPO} vào {proj_path}...")
    subprocess.run(["git", "clone", auth_repo, str(proj_path)], check=True)
    frontend_changed = True

print("✅ Thư mục dự án:", proj_path)

# 3. Tạo môi trường Python 3.11 bằng uv & cài thư viện GPU (tránh xung đột Python 3.13 trên Colab)
print("\n=== 3. Cài đặt môi trường Python 3.11 + GPU (PP-OCRv5 + Faster-Whisper + VietOCR) ===")
venv_dir = Path("/content/venv-submagic")
venv_py = venv_dir / "bin" / "python"
if not venv_py.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", str(venv_dir), "--python", "3.11"], check=True)
    pkgs = [
        "fastapi>=0.115.0", "uvicorn[standard]>=0.32.0", "python-multipart>=0.0.12",
        "pydantic>=2.9.0", "pydantic-settings>=2.6.0", "aiosqlite>=0.20.0",
        "faster-whisper>=1.1.0", "ctranslate2>=4.5.0",
        "onnxruntime-gpu>=1.19.0", "rapidocr-onnxruntime>=1.3.24",
        "opencv-python-headless>=4.10.0.84", "numpy>=1.26.0,<2.0", "Pillow>=10.4.0",
        "rapidfuzz>=3.10.0", "google-genai>=1.0.0", "httpx>=0.27.0",
        "websockets>=13.1", "imageio-ffmpeg>=0.5.1", "av>=13.0.0",
        "torch>=2.2.0", "torchvision>=0.17.0", "vietocr>=0.3.8"
    ]
    subprocess.run(["uv", "pip", "install", "--python", str(venv_py), *pkgs], check=True)
    subprocess.run(["uv", "pip", "uninstall", "--python", str(venv_py), "onnxruntime"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["uv", "pip", "install", "--python", str(venv_py), "--reinstall", "onnxruntime-gpu>=1.19.0"], check=True)
print("✅ Môi trường Python 3.11 + GPU sẵn sàng.")

# 4. Tải model PP-OCRv5 nếu chưa có
models_dir = proj_path / "backend" / "data" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
v5_rec_path = models_dir / "ch_PP-OCRv5_rec_infer.onnx"
if not v5_rec_path.is_file():
    print("⬇️ Đang tải model PP-OCRv5 (15.9 MB)...")
    urllib.request.urlretrieve(
        "https://huggingface.co/breezedeus/cnocr-ppocr-ch_PP-OCRv5/resolve/main/ch_PP-OCRv5_rec_infer.onnx",
        str(v5_rec_path),
    )
print("✅ Model PP-OCRv5 sẵn sàng:", v5_rec_path)

# 5. Build Frontend nếu chưa có dist/ hoặc frontend vừa có commit mới
frontend_dir = proj_path / "frontend"
dist_index = frontend_dir / "dist" / "index.html"
if not dist_index.is_file() or frontend_changed:
    print("🔨 Đang build giao diện React (frontend/dist)...")
    if shutil.which("npm") is None:
        subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs", shell=True, check=True)
    if not (frontend_dir / "node_modules").exists():
        subprocess.run(["npm", "install", "--silent"], cwd=str(frontend_dir), check=True)
    subprocess.run(["npm", "run", "build"], cwd=str(frontend_dir), check=True)
print("✅ Frontend SPA sẵn sàng:", dist_index)

# 6. Khởi động Backend (kèm Frontend SPA) & Tunnel công khai
print("\n=== 4. Khởi động Server & Tạo link truy cập ===")
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)
time.sleep(1)

backend_log = open("/tmp/submagic_backend.log", "w")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
uvicorn_proc = subprocess.Popen(
    [str(venv_py), "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=str(proj_path / "backend"),
    env=env,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Backend không khởi động được. Xem log tại /tmp/submagic_backend.log")

cf_bin = Path("/tmp/cloudflared")
if not cf_bin.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cf_bin),
    )
    os.chmod(str(cf_bin), 0o755)

cf_log = open("/tmp/cloudflared.log", "w+")
cf_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(25):
    time.sleep(1)
    cf_log.seek(0)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", cf_log.read())
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 60)
print("🎉 SUBMAGIC ĐÃ SẴN SÀNG TRÊN GPU T4!")
if public_url:
    print(f"🌐 Link mở trình duyệt (khuyên dùng): {public_url}")
try:
    from google.colab.output import eval_js
    colab_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print(f"🔗 Link nội bộ Colab (dự phòng):   {colab_url}")
except Exception:
    pass
print("=" * 60)

In [ ]:
#@title 🛑 BƯỚC 2: Dừng Server (chỉ chạy khi muốn tắt)
import subprocess
subprocess.run(["pkill", "-f", "uvicorn app.main:app"])
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
print("✅ Đã dừng Submagic server.")